# Interactive Exercise: Develop a Baseline Forecast

In this exercise, you will use historical Illinois General Funds expenditure data to develop two simple baseline forecasts:

1. Carry-forward forecast
2. Average growth rate forecast

By the end of the exercise, you will be able to:

- Calculate annual expenditure growth rates.
- Develop two baseline forecasts.
- Compare the assumptions and results of each method.
- Visualize historical expenditures and forecast values.
- Select the method that appears most appropriate for an expenditure category.

> **Milestone:** This exercise produces your first expenditure forecast.

## 1. Load the Required Packages

The following packages are used to import, organize, and visualize the data.

In [ ]:
library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(scales)
library(tibble)

## 2. Load the Expenditure Data

The sample dataset contains Illinois General Funds expenditures for major agencies and expenditure categories from FY2016 through FY2025.

The notebook uses a relative file path so that the code works both locally and in Binder.

In [ ]:
df <- read_csv("../data/illinois_expenditures.csv")

df

## 3. Review the Available Expenditure Series

The `series` column contains short names that can be used in the code. The `label` column contains the full expenditure category name.

In [ ]:
df %>%
  select(series, label)

## 4. Select an Expenditure Category

Change the value below to forecast a different expenditure category.

Examples include:

- `"education"`
- `"healthcare"`
- `"human_services"`
- `"higher_ed"`
- `"corrections"`
- `"aging"`
- `"total_expenditures"`

In [ ]:
selected_series <- "education"

## 5. Prepare the Selected Series

The source dataset is stored in wide format, with one column for each fiscal year. The following code selects one expenditure category and converts its annual values into a format suitable for calculations and plotting.

In [ ]:
selected_df <- df %>%
  filter(series == selected_series)

if (nrow(selected_df) == 0) {
  stop(
    paste(
      "The selected series was not found.",
      "Choose one of the values listed in the series column."
    )
  )
}

plot_df <- selected_df %>%
  pivot_longer(
    cols = starts_with("fy"),
    names_to = "fiscal_year",
    values_to = "expenditure"
  ) %>%
  mutate(
    fiscal_year = as.numeric(gsub("fy", "", fiscal_year)) + 2000
  ) %>%
  arrange(fiscal_year)

plot_df

## 6. Calculate Annual Growth Rates

The annual growth rate measures the percentage change in expenditure from one fiscal year to the next:

$$
g_t = \frac{Y_t-Y_{t-1}}{Y_{t-1}}
$$

The first year does not have a growth rate because there is no prior observation.

In [ ]:
plot_df <- plot_df %>%
  mutate(
    annual_growth = expenditure / lag(expenditure) - 1
  )

growth_table <- plot_df %>%
  transmute(
    fiscal_year,
    expenditure,
    annual_growth_percent = annual_growth * 100
  )

growth_table

## 7. Summarize Historical Growth

The average annual growth rate will be used to construct the second baseline forecast.

Because unusually high or low years can influence the average, the median growth rate is also reported for comparison.

In [ ]:
average_growth_rate <- mean(
  plot_df$annual_growth,
  na.rm = TRUE
)

median_growth_rate <- median(
  plot_df$annual_growth,
  na.rm = TRUE
)

growth_summary <- tibble(
  measure = c(
    "Average annual growth rate",
    "Median annual growth rate"
  ),
  value = c(
    average_growth_rate,
    median_growth_rate
  )
) %>%
  mutate(
    value_percent = percent(value, accuracy = 0.1)
  )

growth_summary

## 8. Method 1: Carry-Forward Forecast

The carry-forward method assumes that expenditure in the next fiscal year will equal the most recent observed expenditure:

$$
\hat{Y}_{t+1}=Y_t
$$

This method is most appropriate when expenditures are relatively stable and no major change is expected.

In [ ]:
latest_year <- max(plot_df$fiscal_year)

forecast_year <- latest_year + 1

latest_expenditure <- plot_df %>%
  filter(fiscal_year == latest_year) %>%
  pull(expenditure)

carry_forward_forecast <- latest_expenditure

carry_forward_forecast

## 9. Method 2: Average Growth Rate Forecast

The average growth rate method applies the historical average annual growth rate to the most recent expenditure:

$$
\hat{Y}_{t+1}=Y_t(1+g)
$$

This method is useful when historical expenditure growth has been reasonably consistent.

In [ ]:
average_growth_forecast <- latest_expenditure *
  (1 + average_growth_rate)

average_growth_forecast

## 10. Compare the Baseline Forecasts

The following table compares the two forecast methods.

In [ ]:
forecast_comparison <- tibble(
  method = c(
    "Carry-forward",
    "Average growth rate"
  ),
  forecast_year = forecast_year,
  forecast_expenditure = c(
    carry_forward_forecast,
    average_growth_forecast
  )
) %>%
  mutate(
    forecast_expenditure = round(
      forecast_expenditure,
      1
    )
  )

forecast_comparison

## 11. Visualize the Forecasts

The chart below displays:

- Historical expenditures.
- The carry-forward forecast.
- The average growth rate forecast.

The two forecast points illustrate how different assumptions produce different estimates for the same fiscal year.

In [ ]:
historical_plot_df <- plot_df %>%
  transmute(
    fiscal_year,
    expenditure,
    observation_type = "Historical"
  )

forecast_plot_df <- tibble(
  fiscal_year = c(
    forecast_year,
    forecast_year
  ),
  expenditure = c(
    carry_forward_forecast,
    average_growth_forecast
  ),
  observation_type = c(
    "Carry-forward forecast",
    "Average growth forecast"
  )
)

forecast_plot <- ggplot() +
  geom_line(
    data = historical_plot_df,
    aes(
      x = fiscal_year,
      y = expenditure
    ),
    linewidth = 1.1,
    color = "#2F5D8A"
  ) +
  geom_point(
    data = historical_plot_df,
    aes(
      x = fiscal_year,
      y = expenditure
    ),
    size = 3,
    color = "#2F5D8A"
  ) +
  geom_segment(
    aes(
      x = latest_year,
      xend = forecast_year,
      y = latest_expenditure,
      yend = carry_forward_forecast
    ),
    linewidth = 1,
    linetype = "dashed",
    color = "#6A6A6A"
  ) +
  geom_segment(
    aes(
      x = latest_year,
      xend = forecast_year,
      y = latest_expenditure,
      yend = average_growth_forecast
    ),
    linewidth = 1,
    linetype = "dashed",
    color = "#A05A2C"
  ) +
  geom_point(
    data = forecast_plot_df,
    aes(
      x = fiscal_year,
      y = expenditure,
      shape = observation_type
    ),
    size = 4
  ) +
  scale_shape_manual(
    values = c(
      "Carry-forward forecast" = 15,
      "Average growth forecast" = 17
    )
  ) +
  scale_x_continuous(
    breaks = c(
      plot_df$fiscal_year,
      forecast_year
    ),
    labels = paste0(
      "FY",
      c(
        plot_df$fiscal_year,
        forecast_year
      )
    )
  ) +
  scale_y_continuous(
    labels = comma
  ) +
  labs(
    title = paste(
      "Baseline Forecast:",
      unique(plot_df$label)
    ),
    subtitle = paste0(
      "Historical expenditures through FY",
      latest_year,
      " and baseline forecasts for FY",
      forecast_year
    ),
    x = "Fiscal Year",
    y = "Expenditure ($ millions)",
    shape = NULL
  ) +
  theme_minimal(
    base_size = 13
  ) +
  theme(
    plot.title = element_text(
      face = "bold"
    ),
    panel.grid.major.x = element_blank(),
    panel.grid.minor = element_blank(),
    legend.position = "bottom"
  )

forecast_plot

## 12. Try Another Expenditure Category

Return to the `selected_series` cell and replace `"education"` with another value.

For example:

```r
selected_series <- "healthcare"
```

or:

```r
selected_series <- "corrections"
```

Then rerun the cells below that selection.

Compare how the baseline forecasts change across expenditure categories.

## Questions for Reflection

1. Which method produces the larger forecast?

2. Why do the two methods produce different results?

3. Has the selected expenditure category experienced relatively consistent historical growth?

4. Are there any unusual years that may be affecting the average growth rate?

5. Which baseline method appears more appropriate for this category?

6. Are there policy, demographic, or economic factors that should be incorporated before using the forecast for budget planning?

7. Would a more sophisticated forecasting method likely improve the forecast?

## 13. Optional: Save the Forecast Figure

The following code saves the chart in an `output` folder within the exercise repository.

Binder sessions are temporary. Download any files you wish to retain before closing the session.

In [ ]:
dir.create(
  "../output",
  showWarnings = FALSE
)

output_filename <- paste0(
  "../output/",
  selected_series,
  "_baseline_forecast.png"
)

ggsave(
  filename = output_filename,
  plot = forecast_plot,
  width = 8,
  height = 5,
  dpi = 300
)

output_filename

## Conclusion

You have now developed two transparent baseline forecasts.

These forecasts provide a benchmark for evaluating whether more advanced methods add meaningful forecasting value.

In the next building block, time series methods will be used to model historical trends and other time-dependent patterns more formally.